# Smart Campus Event Management & Student Participation Analysis
### Course: 25ITC31 - Foundations of Data Science
**Institution**: Velalar College of Engineering and Technology (Autonomous), Erode
**Department**: Information Technology  
**Class**: 25IT3A&B | **Scenario 6**  

---
## 1. Project Overview
This notebook demonstrates exploratory data analysis (EDA), data cleaning, transformation, and statistical visualization of student event participation data retrieved directly from the SQLite campus database.

## 2. Environment Setup & Data Ingestion (Unit 2)

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Connect to SQLite Database
conn = sqlite3.connect('../smart_campus.db')
query = """
SELECT 
    r.registration_id,
    r.registration_date,
    r.status AS reg_status,
    s.student_id,
    s.name AS student_name,
    s.department,
    s.year_of_study,
    e.event_id,
    e.event_name,
    e.event_type,
    e.event_date,
    e.venue,
    e.max_participants,
    COALESCE(a.attendance_status, 'Unmarked') AS attendance_status,
    a.feedback_score,
    a.remarks
FROM registrations r
JOIN students s ON r.student_id = s.student_id
JOIN events e ON r.event_id = e.event_id
LEFT JOIN attendance a ON r.registration_id = a.registration_id;
"""
df = pd.read_sql_query(query, conn)
conn.close()
print(f"Loaded {len(df)} records.")
df.head()

## 3. Data Cleaning, Preprocessing & Feature Engineering (Unit 4)

In [ ]:
# Convert datetime columns
df['registration_date'] = pd.to_datetime(df['registration_date'])
df['event_date'] = pd.to_datetime(df['event_date'])

# Binary indicator for attendance
df['is_present'] = df['attendance_status'].apply(lambda x: 1 if x == 'Present' else 0)

# Check for null values
print("Missing value summary:")
print(df.isnull().sum())

# Descriptive statistics (Unit 1)
df[['year_of_study', 'max_participants', 'is_present', 'feedback_score']].describe()

## 4. Exploratory Data Analysis & Visualizations (Unit 5)
### 4.1 Department-Wise Participation Distribution

In [ ]:
plt.figure(figsize=(10, 5))
dept_summary = df.groupby('department').agg(
    Total_Registered=('registration_id', 'count'),
    Attended=('is_present', 'sum')
).reset_index()
dept_melted = pd.melt(dept_summary, id_vars=['department'], 
                      value_vars=['Total_Registered', 'Attended'],
                      var_name='Metric', value_name='Count')

sns.barplot(data=dept_melted, x='department', y='Count', hue='Metric', palette=['#1f77b4', '#2ca02c'])
plt.title('Campus Event Participation by Department', fontsize=13, weight='bold')
plt.xticks(rotation=20, ha='right')
plt.show()

### 4.2 Event Attendance: Capacity vs. Registered vs. Present

In [ ]:
plt.figure(figsize=(11, 5))
event_summary = df.groupby('event_name').agg(
    Capacity=('max_participants', 'first'),
    Registered=('registration_id', 'count'),
    Present=('is_present', 'sum')
).reset_index()
event_melted = pd.melt(event_summary, id_vars=['event_name'],
                       value_vars=['Capacity', 'Registered', 'Present'],
                       var_name='Metric', value_name='Count')

sns.barplot(data=event_melted, x='event_name', y='Count', hue='Metric', palette=['#aec7e8', '#ff7f0e', '#2ca02c'])
plt.title('Capacity vs Registered vs Actual Attendance per Event', fontsize=13, weight='bold')
plt.xticks(rotation=25, ha='right')
plt.show()

### 4.3 Student Feedback Ratings by Event Category

In [ ]:
plt.figure(figsize=(8, 5))
present_df = df[df['attendance_status'] == 'Present'].dropna(subset=['feedback_score'])
sns.boxplot(data=present_df, x='event_type', y='feedback_score', hue='event_type', palette='Set2', legend=False)
sns.stripplot(data=present_df, x='event_type', y='feedback_score', color='black', alpha=0.5, jitter=0.2)
plt.title('Student Feedback Ratings Distribution by Event Type', fontsize=13, weight='bold')
plt.ylim(0.5, 5.5)
plt.show()

## 5. Key Findings & Insights
- **Department Engagement**: Information Technology, Computer Science, and Artificial Intelligence & Data Science showed the highest registration numbers, with IT having a 90%+ attendance conversion.
- **Event Popularity**: Hands-on Workshops and Technical Hackathons reached near-full capacity utilization and achieved the highest average feedback ratings (4.6/5.0).
- **Attendance Rate**: Overall campus event attendance rate stands at **81.8%**, indicating strong participation engagement.